# Fusing the Exponential into the Multiply: an Open-Source Reproduction of ExpMul for FlashAttention-2

### 4 RTL variants through RTL-to-GDS on an open 45 nm library · 2 000 000 operator samples · FA-2 kernel, GLUE and post-route power

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Guo-4869/FlashAttention_Silicon/blob/main/flash_attn_expmul.ipynb) (repository notebook) ·
[![GitHub](https://img.shields.io/badge/GitHub-FlashAttention__Silicon-181717?logo=github)](https://github.com/Guo-4869/FlashAttention_Silicon)

**Paper reproduced:** [arXiv:2505.14314](https://arxiv.org/abs/2505.14314) ·
**Reference implementation:** [ic-lab-duth/Fused-ExpMul](https://github.com/ic-lab-duth/Fused-ExpMul) (MIT) ·
**This notebook:** [GitHub](https://github.com/Guo-4869/FlashAttention_Silicon/blob/main/flash_attn_expmul.ipynb)

_Submission track: **&lt;fill in: venue / track / date&gt;**_

| Name | Affiliation | IEEE Member | SSCS Member | Email |
|---|---|---|---|---|
| _&lt;your name&gt;_ (Leader) | _&lt;affiliation&gt;_ | _&lt;Yes / No&gt;_ | _&lt;Yes / No&gt;_ | _&lt;email&gt;_ |
| _&lt;advisor name&gt;_ (Advisor) | _&lt;affiliation&gt;_ | _&lt;Yes / No&gt;_ | _&lt;Yes / No&gt;_ | _&lt;email&gt;_ |

> **To fill in before submission:** the author table above, the submission track line, and any
> acknowledgement. Everything else is generated by the code in this repository.

`SPDX-License-Identifier: MIT` · Tools: MSVC / g++, Python 3.12 + NumPy, Icarus Verilog 12,
Yosys 0.68 + ABC, OpenROAD (OpenROAD-flow-scripts), nangate45 open-cell library (45 nm).

## Abstract

FlashAttention-2 computes attention online and keeps only a running maximum, a running sum of
exponentials and a running output. The paper under reproduction observes that the scalar
exponential and the two vector multiplications of the update can be fused into one operator,
`ExpMul(x, V) = e^x·V`, which replaces a floating-point exponential unit and a floating-point
multiplier with an integer shift-and-add on a Q6.10 fixed-point exponent and an exponent-field
subtraction, returning a floating-point result without any dequantisation step. This reproduction
rebuilds the operator in a bit-accurate C++/NumPy model, integrates it into a FlashAttention-2
kernel with an unchanged loop order, evaluates it on Flan-T5 over GLUE, and takes four RTL variants
(baseline and ExpMul, FP32 and BF16) through synthesis, place-and-route and switching-activity-based
power on an open 45 nm library. The measurements reproduce the direction and the order of magnitude
of the paper, and identify one point where the textual algorithm and the released implementation
differ by an order of magnitude in accuracy.

| 2 000 000 | 4 + 4 | **−69.3 % / −70.2 %** | **3.4 pp** |
|:---:|:---:|:---:|:---:|
| random (x, V) pairs for the<br>operator error study | FA-2 kernel error at N=512,<br>FP32: baseline 0.08 % / ExpMul 5.1 % | lane area / power saving,<br>FP32, post-route + VCD | spread between the two<br>readings of Algorithm 3 |

## 1. Why reproduce the operator, not just the kernel

Attention hardware spends a large fraction of its area and energy on the exponential and on the
multiply that immediately follows it. The paper's claim is that this pair can be replaced by
integer arithmetic with no loss of model accuracy, and that the saving is visible after full
place-and-route. Three things are therefore worth checking independently:

* whether the operator itself is accurate enough, and how its error propagates through the online
  softmax normalisation, which is the only place where the error can cancel;
* whether the block-level area saving survives synthesis and place-and-route, where fixed overheads
  dilute the saving of a replaced datapath;
* whether the "no accuracy loss" claim holds when the operator is used inside a real language model.

A fourth question arose while reading the code: Algorithm 3 in the paper updates only the exponent
field, whereas the released implementation also folds the fractional residual into the mantissa.
The two differ by about an order of magnitude in accuracy, and the difference decides whether the
language-model claim is plausible. This reproduction implements and evaluates both.

## 2. How the measurement works

| | |
|---|---|
| **Operator model** | Q6.10 fixed point of `Clip(x, -15, 0)`, shift-and-add `L = round(-x·log2 e)`, exponent-field update; two variants (exponent-only, mantissa-interpolated) plus a `ceil` ablation |
| **Kernel** | FlashAttention-2 with the exact loop order of Algorithm 2, `sigma* = [l, o]` merge of Eq. (3), and `P` query blocks with broadcast K/V; baseline and ExpMul share one driver and differ only in the update functor |
| **References** | double-precision FA-2 with the same loop order for the kernel; exact `e^x·V` for the operator |
| **Language model** | `google/flan-t5-base`, GLUE validation split, 100 examples per task, teacher-forced label scoring (no decoding) |
| **RTL** | hand-written synthesizable Verilog: IEEE-754 binary32 (and BF16) multiply/add, ExpMul, 8-segment piecewise-linear exponential, one query block with a pipelined adder tree and a single-cycle update |
| **Implementation** | Yosys + ABC for area; OpenROAD-flow-scripts for floorplan, placement, CTS, routing and GDS; nangate45 (45 nm, 1.1 V) |
| **Power** | behavioural library models generated from Liberty, 300 random cycles on the post-route netlist, VCD annotated in OpenROAD together with the extracted SPEF |
| **Determinism** | seeds 20250520 (sweeps), 777 (stability), 424242 (per-query detail); fixed tool versions listed in §8 |

## 3. Result 1 — the operator is accurate only when the residual reaches the mantissa

Two million random pairs, with `x` drawn from a mixture of uniform distributions on `[-15, 0]` and
`V` with random sign, exponent and mantissa. Errors are relative to the double-precision value of
`e^x·V`.

| Variant | mean | RMS | p90 | p99 | max | within 5 % |
|---|---|---|---|---|---|---|
| Exponent-only (Algorithm 3 as written) | +0.0000 | **0.2117** | 0.3412 | 0.4298 | 0.4925 | 14.1 % |
| Exponent-only, `ceil` rounding | +0.0003 | 0.3091 | 0.4572 | 0.4913 | 0.4998 | 9.1 % |
| Mantissa-interpolated (released code) | −0.0000 | **0.0331** | 0.0547 | 0.0855 | 0.1202 | 86.7 % |

![Weight quantisation and relative error](results/phase1/phase1_weight_ratio.png)

![Error distribution over 2 M random pairs](results/phase1/phase1_error_hist.png)

**What the figure says.** The exponent-only error is structural, not a rounding bug: the weight is
quantised to an integer power of two, so the ratio is confined to `[2^-0.5, 2^0.5]`, and an RMS of
21 % is the best this formulation can do. The mantissa-interpolated form uses the same integer
shift-and-add but also adds the fractional residual to the integer representation of `V`;
the result is a piecewise-linear approximation of `2^-f` with a 3.3 % RMS error (theoretical
maximum about 11 %). Boundary behaviour is exact where it matters: `x = 0` returns `V` bit-for-bit,
`x = -15` saturates at `2^-22`, positive inputs are clipped to zero (no positive exponent
overflow), and no NaN or infinity is produced in any random test.

## 4. Result 2 — inside the kernel, the normalisation absorbs part of the error, not all of it

Per-query relative L2 error against the double-precision FA-2 reference, `d = 64`, temperature 1:

| Implementation | N=16 | N=64 | N=128 | N=512 |
|---|---|---|---|---|
| Baseline FP32 | 0.05 % | 0.07 % | 0.07 % | **0.08 %** |
| ExpMul, exponent-only | 12.3 % | 19.7 % | 18.6 % | 22.5 % |
| ExpMul, mantissa-interpolated | 3.3 % | 4.5 % | 4.6 % | **5.1 %** |
| Baseline BF16 | 0.69 % | 1.16 % | 1.70 % | 14.4 % |
| ExpMul BF16, mantissa-interpolated | 2.68 % | 4.38 % | 4.49 % | 12.0 % |

![Kernel error versus sequence length](results/phase2/phase2_error_vs_N.png)

![Per-query error distribution](results/phase2/phase2_error_hist.png)

**What the kernel shows.** The `o/l` normalisation cancels a common scaling but not an
element-wise weight error, so the 21 % operator error survives as 12–22 % at the output, while the
mantissa-interpolated variant stays at 3–5 % (its error accumulates multiplicatively on the `o`
accumulator rather than coming from a single operator call). Under BF16 the ordering inverts at
N = 512: the baseline is worse than ExpMul because rounding each PWL exponential to bf16 is random
noise, whereas the logarithmic quantisation of ExpMul is a smooth bias that partly cancels in
`o/l`.

**Stability.** With scores scaled to 1e3 and 1e6 all three implementations agree to about 1e-5 in
absolute terms, while a naive softmax without max subtraction overflows in FP32; no NaN or infinity
occurs in any case, and the block-parallel configuration (`P = 4`) is bit-identical to the
sequential one (`P = 1`).

![Stability cases](results/phase2/phase2_stability.png)

## 5. Result 3 — on Flan-T5 the variants are indistinguishable at this sample size

Every `T5Attention.forward` is replaced by the FA-2 kernel with ExpMul weights; the rest of the
model is untouched.

| Task | FP32 | FP32 exponent-only | FP32 mantissa | BF16 | BF16 mantissa |
|---|---|---|---|---|---|
| SST-2 (accuracy) | 81.0 | 79.0 | **82.0** | 81.0 | 81.0 |
| MNLI-m (accuracy) | 37.0 | 36.0 | 37.0 | 37.0 | 37.0 |
| MNLI-mm (accuracy) | 28.0 | 29.0 | 29.0 | 29.0 | 29.0 |

![GLUE accuracy](results/phase3/phase3_glue.png)

**What the table says, and what it does not.** The differences between variants are at most two
points, while the 95 % binomial interval for 100 examples is about ±8 points: the comparison is
indicative, not conclusive. The absolute levels are also not comparable with the paper's Table I,
which uses the few-shot PromptBench protocol and a larger Flan-T5; here the zero-shot base model
reaches 37 % on three-way MNLI (random is 33 %), a clear prompt/scale mismatch. The direction,
however, agrees with the kernel study: the mantissa-interpolated variant tracks FP32, the
exponent-only variant is lower. A ±1 % claim would require `flan-t5-large`/`xl`, few-shot prompts
and at least 1000 examples per task.

## 6. Result 4 — the area saving survives place-and-route; the power saving follows it

Primitive areas after synthesis (nangate45, µm²): FP32 multiply 4 069.5, FP32 add 1 519.7,
PWL exponential 4 320.9, PWL reciprocal 1 668.4, **ExpMul 699.6** (exponent-only) / 856.0
(mantissa-interpolated). Composing the block from these primitives (dot-product tree, `D` update
lanes, final division):

| d | Baseline (µm²) | ExpMul (µm²) | Saving | Paper (FP32) |
|---|---|---|---|---|
| 16 | 317 870 | 206 395 | **35.1 %** | 23.6 % |
| 32 | 626 949 | 412 640 | **34.2 %** | — |
| 64 | 1 245 107 | 825 132 | **33.7 %** | 29.7 % |
| 256 | 4 954 058 | 3 300 081 | **33.4 %** | 33.1 % |

The saving exceeds the reported 28.8 % because the composed model omits fixed overheads (control,
registers, multiplexing) that the paper's HLS block contains — its own small-`d` result of 23.6 %
shows those overheads dominating there.

Four lane designs were then taken through the complete flow. They use distinct top-level modules,
which is essential: sharing a `DESIGN_NAME` makes ORFS reuse the synthesised database, and an
earlier run silently compared a netlist with itself (18 776 instances on both sides). Verified
fingerprints: 18 776 vs. 7 436 instances for FP32, 6 448 vs. 3 972 for BF16.

| Data type | Variant | Area (µm²) | Saving | Power @ 50 MHz | Saving |
|---|---|---|---|---|---|
| FP32 | baseline | 30 539 | — | 3.42 mW | — |
| FP32 | ExpMul | 9 362 | **−69.3 %** | 1.02 mW | **−70.2 %** |
| BF16 | baseline | 9 041 | — | 0.931 mW | — |
| BF16 | ExpMul | 5 092 | **−43.7 %** | 0.554 mW | **−40.5 %** |

![Area and power, FP32 and BF16](results/phase4/phase4_power.png)

![Block area versus the paper](results/phase4/phase4_area.png)

**Why the two data types differ.** The FP32 variant removes six full-precision multipliers and two
PWL exponentials (two multipliers each), so the lane shrinks by 69 % and its power by 70 %. For
BF16 the multipliers are already small, so replacing them with integer shift-and-add yields a
smaller marginal benefit (−44 % / −40 %); the absolute area of the BF16 ExpMul lane, however, is
only one sixth of the FP32 baseline. A 2 ns constraint cannot be met in this 45 nm library
(worst slack ≈ −15 ns), which is why the runs use a 20 ns period; the paper's 500 MHz target relies
on a 28 nm process.

### Post-route layouts

![Post-route layouts of the four lane designs](results/visuals/layout_compare.png)

The layout comparison shows the same ordering as the numbers: the FP32 baseline (30.5k µm²) is the
densest, the FP32 ExpMul version (9.4k µm²) is visibly sparser, and both BF16 versions are more
compact. GDS files and all 40 ORFS stage images are under `results/visuals/`.

## 7. Where this reproduction departs from the paper

1. **Two readings of Algorithm 3.** The textual formulation changes only the exponent field and
   incurs 15–22 % kernel-level error; the released implementation folds the residual into the
   mantissa and incurs 2–5 %. Only the latter is consistent with the paper's Table I. This is the
   single largest discrepancy found, and it is a specification discrepancy, not a numerical one.
2. **Area saving is higher than reported (34 % vs. 28.8 %).** The composed block model omits the
   fixed overheads present in the paper's HLS block; the paper's small-`d` value confirms they
   dominate there.
3. **Absolute magnitudes agree.** The paper reports 5.3×10⁵ µm² for `d = 64`, FP32, at 28 nm,
   i.e. about 1.38×10⁶ µm² after scaling to 45 nm; the model gives 1.245×10⁶ µm².
4. **Power is measured at lane level, extrapolated at block level.** The lane saving is −70.2 %
   (FP32) and −40.5 % (BF16); a block-level saving of roughly −35 % is an extrapolation from the
   area share of the update path, not a measurement of a complete kernel.
5. **Language-model evidence is indicative only.** The 100-example protocol cannot resolve
   differences below eight points.

## 8. Reproducibility

| What | How it is produced here |
|---|---|
| Operator error tables (§3) | `./build/expmul results/phase1` (2 M samples, seed 20250520); bit-exact cross-check with `model/expmul_ref.py --check` |
| Kernel error and stability (§4) | `./build/tb results/phase2` (3 variants × 2 dtypes × 3 `d` × 4 `N` × 2 temperatures) |
| GLUE results (§5) | `model/t5_glue.py` — attention replacement, teacher-forced scoring, 100 examples per task |
| Primitive and block area (§6) | `scripts/synth_units.sh` (Yosys + ABC) and `model/area_model.py` |
| Post-route area/power (§6) | `scripts/setup_orfs_*.sh` + OpenROAD-flow-scripts; `scripts/gen_vcd.sh` and `scripts/power_*.sh` (VCD + SPEF) |
| Figures | `model/plot_phase1.py`, `plot_phase2.py`, `plot_phase34.py`, `make_visuals.py` |
| Tool versions | MSVC 19.51 / g++ 13.3, Icarus Verilog 12.0, Yosys 0.68, OpenROAD master, nangate45; Python 3.12, NumPy 2.5 |

```bash
bash scripts/build.sh                 # C++ models
./build/expmul results/phase1 && ./build/tb results/phase2
bash scripts/synth_units.sh && python model/area_model.py
bash scripts/setup_orfs_bf16.sh && bash scripts/power_bf16.sh
venv/bin/python t5_glue.py --model models/flan-t5-base --data_dir data/glue --n 100 \
    --variants fp32 fp32_exp fp32_mi bf16 bf16_mi
```

## 9. Conclusion and limitations

**Conclusion.** The fused operator reproduces the paper's direction and magnitude: an exponent-field
update with an integer shift-and-add removes the floating-point exponential and the following
multiply, cutting a post-route lane from 30 539 to 9 362 µm² (−69.3 %) and from 3.42 to 1.02 mW
(−70.2 %) in FP32, with a BF16 variant at −43.7 % area and −40.5 % power. Inside FlashAttention-2
the operator's error is largely, but not entirely, absorbed by the online normalisation, and only
the mantissa-interpolated reading of Algorithm 3 keeps the kernel error at the few-percent level
needed for an accuracy-neutral substitution.

**Limitations.** One open PDK (45 nm), a lane-level power measurement with a block-level
extrapolation, a 100-example GLUE protocol that cannot resolve sub-eight-point differences, and a
BF16 datapath whose power advantage is smaller than the FP32 one. The complete kernel has not yet
been taken through place-and-route as a single netlist, and the language-model study uses the base
model rather than the larger Flan-T5 of the paper.

## References

1. K. Alexandridis, V. Titopoulos, G. Dimitrakopoulos, *Low-Cost FlashAttention with Fused
   Exponential and Multiplication Hardware Operators*, [arXiv:2505.14314](https://arxiv.org/abs/2505.14314), 2025.
   Reference implementation: [github.com/ic-lab-duth/Fused-ExpMul](https://github.com/ic-lab-duth/Fused-ExpMul) (MIT).
2. T. Dao, *FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*,
   ICLR 2024, [arXiv:2307.08691](https://arxiv.org/abs/2307.08691).
3. W. Wang et al., *SOLE: Hardware-Software Co-design of Softmax and LayerNorm for Efficient
   Transformer Inference*, ICCAD 2023 — source of the 1.4375 shift-and-add constant.
4. C. Raffel et al., *Exploring the Limits of Transfer Learning with a Unified Text-to-Text
   Transformer*, JMLR 21(140), 2020.
5. H. W. Chung et al., *Scaling Instruction-Finetuned Language Models*, JMLR 25(70), 2024.
6. A. Wang et al., *GLUE: A Multi-Task Benchmark and Analysis Platform for Natural Language
   Understanding*, [arXiv:1804.07461](https://arxiv.org/abs/1804.07461), 2018.
7. K. Zhu et al., *PromptBench: A Unified Library for Evaluation of Large Language Models*,
   [arXiv:2312.07910](https://arxiv.org/abs/2312.07910), 2023.